## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 23 samples and 11724 columns.
First few rows of the data:


,overall survival time (os in months),death,recurrence free survival time (rfs in months),event,samples,diagnosis,tissue type,study group,treatment,SCYL3,...,USP51,ZNF674,KDM5D,DDX3Y,ZFY,USP9Y,RPS4Y1,NLGN4Y,UTY,EIF1AY
0,64.5667,0,12.30,1,Sondel_RNA_A,melanoma,tumor,A,Hu14.18-IL2,2.540495,...,1.072710,2.435068,11.532765,90.812886,20.731276,28.482075,529.672849,17.286358,19.622720,11.087398
1,31.2333,1,0.76,1,Sondel_RNA_B,melanoma,tumor,B,Hu14.18-IL2,2.259527,...,1.780658,2.116987,0.000000,0.055146,0.022062,0.047871,0.125179,0.019280,0.003478,0.000000
2,9.9000,1,0.00,1,Sondel_RNA_C,melanoma,tumor,B,Never Treated,2.744963,...,2.915865,1.182428,13.912200,52.902504,20.271464,44.052409,189.020923,16.856049,22.095420,8.046334
3,53.0667,0,48.76,0,Sondel_RNA_D,melanoma,tumor,B,Hu14.18-IL2,3.266027,...,0.986413,3.671449,21.166866,29.791873,20.176599,38.073402,347.483597,1.953853,24.797011,5.469122
4,9.4667,1,0.16,1,Sondel_RNA_E,melanoma,tumor,A,Hu14.18-IL2,3.014413,...,0.511402,2.174036,0.003904,0.025756,0.038640,0.046031,0.373527,0.000000,0.109634,0.009069


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'samples'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
